## Retraining `mu_delta`'s source model with an active-player training filter

Two changes were bundled together in the earlier `total_points`-targeted retrain of this
pipeline (`hist/{pos}_preds_points.csv`): (a) switching the prediction target from `xP` to
`total_points`, and (b) filtering the TRAINING pool down to players with recent minutes
(`minutes_rolling_3`/`minutes_rolling_5` not both zero/NaN-as-unknown), so long-term-inactive
players don't dilute feature selection and model fitting. That combination showed a
*better* holdout R2 in isolation but made the full rolling-optimization backtest markedly
*worse* (1332-1413 vs 1765 real-history points, depending on the fixture tilt) than the
original `xP`-targeted model it replaced.

This retrains against `xP` again (the original target, matching `hist/{pos}_preds.csv` and
what `rolling_optimization.ipynb`/`optimization.ipynb` currently read) but keeps the
active-player training filter -- isolating whether that filter alone helps, independent of
the target-column change that was confounded with it before. Produces new, separate
artifacts (`hist/feats_{pos}_xp_v2`, `models_by_round_{pos}_xp_v2`,
`hist/{pos}_preds_xp_v2.csv`) alongside the existing files rather than overwriting them, so
nothing currently wired into the optimizer changes until this is checked and the notebook
is deliberately repointed.

This notebook mirrors the existing per-position pipeline (`select_features_by_correlation` ->
`evaluate_across_gws` -> `tune_best_model`, walk-forward by round), generalized so the same
code trains either target (set via `TARGET_COL`/`OUTPUT_SUFFIX` below) -- it was previously
used to produce the `total_points`-targeted run (`OUTPUT_SUFFIX='points'`) and is reused here
unchanged for this `xP`-targeted run (`OUTPUT_SUFFIX='xp_v2'`).


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.ensemble import RandomForestRegressor

import joblib

from utils import evaluate_across_gws, select_features_by_correlation, tune_best_model


In [ ]:
player_data = pd.read_csv('../rolled_data_24_25.csv', index_col=False)

TARGET_COL = 'xP'
# Distinguishes this run's saved artifacts (hist/feats_{pos}_xp_v2, hist/{pos}_preds_xp_v2.csv,
# etc.) from both the original xP-predictor notebooks' files (hist/{pos}_preds.csv, no
# active-player filter) and the earlier total_points-targeted run (OUTPUT_SUFFIX='points').
OUTPUT_SUFFIX = 'xp_v2'
# TARGET_COL is never itself one of the candidate features (it's not in AVAIL_FEATS_BY_POS
# below -- the raw xP column was already excluded there, same reasoning as the docstring
# above about not letting a same-gameweek value leak into its own prediction), so it doesn't
# need listing here as something to strip back out downstream -- only 'round'/'element'
# (identifiers, not predictors) do.
DROP_COLS = (TARGET_COL, 'round', 'element')

POSITIONS = {
    'gk': 'Goalkeeper',
    'def': 'Defender',
    'mid': 'Midfielder',
    'fwd': 'Forward',
}

# Same per-position candidate feature lists as the xP_predictor notebooks, with the raw
# (unlagged) 'xP' column removed from each -- see the markdown above for why.
AVAIL_FEATS_BY_POS = {
    'gk': [
    'was_home',
    'value',
    'transfers_balance',
    'selected',
    'transfers_in',
    'transfers_out',
    'difficulty',
    'opponent_difficulty',
    'win_prob',
    'percentage_net_transfers',
    'elo_diff',
    'ownership_change',
    'points_rolling_1',
    'points_rolling_3',
    'points_rolling_5',
    'xP_rolling_1',
    'xP_rolling_3',
    'xP_rolling_5',
    'minutes_rolling_1',
    'minutes_rolling_3',
    'minutes_rolling_5',
    'clean_sheets_rolling_1',
    'clean_sheets_rolling_3',
    'clean_sheets_rolling_5',
    'goals_conceded_rolling_1',
    'goals_conceded_rolling_3',
    'goals_conceded_rolling_5',
    'yellow_cards_rolling_1',
    'yellow_cards_rolling_3',
    'yellow_cards_rolling_5',
    'saves_rolling_1',
    'saves_rolling_3',
    'saves_rolling_5',
    'influence_rolling_1',
    'influence_rolling_3',
    'influence_rolling_5',
    'creativity_rolling_1',
    'creativity_rolling_3',
    'creativity_rolling_5',
    'threat_rolling_1',
    'threat_rolling_3',
    'threat_rolling_5',
    'ict_index_rolling_1',
    'ict_index_rolling_3',
    'ict_index_rolling_5',
    'starts_rolling_1',
    'starts_rolling_3',
    'starts_rolling_5',
    'expected_goal_involvements_rolling_1',
    'expected_goal_involvements_rolling_3',
    'expected_goal_involvements_rolling_5',
    'expected_goals_conceded_rolling_1',
    'expected_goals_conceded_rolling_3',
    'expected_goals_conceded_rolling_5',
    'interceptions_rolling_1',
    'interceptions_rolling_3',
    'interceptions_rolling_5',
    'blocks_rolling_1',
    'blocks_rolling_3',
    'blocks_rolling_5',
    'clearances_rolling_1',
    'clearances_rolling_3',
    'clearances_rolling_5',
    'tackles_rolling_1',
    'tackles_rolling_3',
    'tackles_rolling_5',
    'chances_created_rolling_1',
    'chances_created_rolling_3',
    'chances_created_rolling_5',
    'goals_prevented_rolling_1',
    'goals_prevented_rolling_3',
    'goals_prevented_rolling_5',
    'sweeper_actions_rolling_1',
    'sweeper_actions_rolling_3',
    'sweeper_actions_rolling_5',
    'tackles_won_percent_rolling_1',
    'tackles_won_percent_rolling_3',
    'tackles_won_percent_rolling_5',
    'opponent_xG_rolling_1',
    'opponent_xG_rolling_3',
    'opponent_xG_rolling_5',
    'opponent_xGOT_rolling_1',
    'opponent_xGOT_rolling_3',
    'opponent_xGOT_rolling_5',
    'shots_faced_rolling_1',
    'shots_faced_rolling_3',
    'shots_faced_rolling_5',
    'opponent_shots_faced_rolling_1',
    'opponent_shots_faced_rolling_3',
    'opponent_shots_faced_rolling_5',
    'big_chances_faced_rolling_1',
    'big_chances_faced_rolling_3',
    'big_chances_faced_rolling_5',
    'opponent_big_chances_faced_rolling_1',
    'opponent_big_chances_faced_rolling_3',
    'opponent_big_chances_faced_rolling_5',
    'total_points_per_90_rolling_1',
    'total_points_per_90_rolling_3',
    'total_points_per_90_rolling_5',
    'clean_sheets_per_90_rolling_1',
    'clean_sheets_per_90_rolling_3',
    'clean_sheets_per_90_rolling_5',
    'goals_conceded_per_90_rolling_1',
    'goals_conceded_per_90_rolling_3',
    'goals_conceded_per_90_rolling_5',
    'saves_per_90_rolling_1',
    'saves_per_90_rolling_3',
    'saves_per_90_rolling_5',
    'expected_goal_involvements_per_90_rolling_1',
    'expected_goal_involvements_per_90_rolling_3',
    'expected_goal_involvements_per_90_rolling_5',
    'expected_goals_conceded_per_90_rolling_1',
    'expected_goals_conceded_per_90_rolling_3',
    'expected_goals_conceded_per_90_rolling_5',
    'interceptions_per_90_rolling_1',
    'interceptions_per_90_rolling_3',
    'interceptions_per_90_rolling_5',
    'blocks_per_90_rolling_1',
    'blocks_per_90_rolling_3',
    'blocks_per_90_rolling_5',
    'clearances_per_90_rolling_1',
    'clearances_per_90_rolling_3',
    'clearances_per_90_rolling_5',
    'tackles_per_90_rolling_1',
    'tackles_per_90_rolling_3',
    'tackles_per_90_rolling_5',
    'chances_created_per_90_rolling_1',
    'chances_created_per_90_rolling_3',
    'chances_created_per_90_rolling_5',
    'goals_prevented_per_90_rolling_1',
    'goals_prevented_per_90_rolling_3',
    'goals_prevented_per_90_rolling_5',
    'sweeper_actions_per_90_rolling_1',
    'sweeper_actions_per_90_rolling_3',
    'sweeper_actions_per_90_rolling_5',
    'tackles_won_percent_per_90_rolling_1',
    'tackles_won_percent_per_90_rolling_3',
    'tackles_won_percent_per_90_rolling_5',
    'points_1_ewm',
    'points_3_ewm',
    'points_5_ewm',
    'xP_1_ewm',
    'xP_3_ewm',
    'xP_5_ewm',
    'minutes_1_ewm',
    'minutes_3_ewm',
    'minutes_5_ewm',
    'clean_sheets_1_ewm',
    'clean_sheets_3_ewm',
    'clean_sheets_5_ewm',
    'goals_conceded_1_ewm',
    'goals_conceded_3_ewm',
    'goals_conceded_5_ewm',
    'yellow_cards_1_ewm',
    'yellow_cards_3_ewm',
    'yellow_cards_5_ewm',
    'saves_1_ewm',
    'saves_3_ewm',
    'saves_5_ewm',
    'influence_1_ewm',
    'influence_3_ewm',
    'influence_5_ewm',
    'creativity_1_ewm',
    'creativity_3_ewm',
    'creativity_5_ewm',
    'threat_1_ewm',
    'threat_3_ewm',
    'threat_5_ewm',
    'ict_index_1_ewm',
    'ict_index_3_ewm',
    'ict_index_5_ewm',
    'starts_1_ewm',
    'starts_3_ewm',
    'starts_5_ewm',
    'expected_goal_involvements_1_ewm',
    'expected_goal_involvements_3_ewm',
    'expected_goal_involvements_5_ewm',
    'expected_goals_conceded_1_ewm',
    'expected_goals_conceded_3_ewm',
    'expected_goals_conceded_5_ewm',
    'interceptions_1_ewm',
    'interceptions_3_ewm',
    'interceptions_5_ewm',
    'blocks_1_ewm',
    'blocks_3_ewm',
    'blocks_5_ewm',
    'clearances_1_ewm',
    'clearances_3_ewm',
    'clearances_5_ewm',
    'tackles_1_ewm',
    'tackles_3_ewm',
    'tackles_5_ewm',
    'chances_created_1_ewm',
    'chances_created_3_ewm',
    'chances_created_5_ewm',
    'goals_prevented_1_ewm',
    'goals_prevented_3_ewm',
    'goals_prevented_5_ewm',
    'sweeper_actions_1_ewm',
    'sweeper_actions_3_ewm',
    'sweeper_actions_5_ewm',
    'tackles_won_percent_1_ewm',
    'tackles_won_percent_3_ewm',
    'tackles_won_percent_5_ewm',
    'opponent_xG_1_ewm',
    'opponent_xG_3_ewm',
    'opponent_xG_5_ewm',
    'opponent_xGOT_1_ewm',
    'opponent_xGOT_3_ewm',
    'opponent_xGOT_5_ewm',
    'shots_faced_1_ewm',
    'shots_faced_3_ewm',
    'shots_faced_5_ewm',
    'opponent_shots_faced_1_ewm',
    'opponent_shots_faced_3_ewm',
    'opponent_shots_faced_5_ewm',
    'big_chances_faced_1_ewm',
    'big_chances_faced_3_ewm',
    'big_chances_faced_5_ewm',
    'opponent_big_chances_faced_1_ewm',
    'opponent_big_chances_faced_3_ewm',
    'opponent_big_chances_faced_5_ewm',
    'total_points_per_90_1_ewm',
    'total_points_per_90_3_ewm',
    'total_points_per_90_5_ewm',
    'clean_sheets_per_90_1_ewm',
    'clean_sheets_per_90_3_ewm',
    'clean_sheets_per_90_5_ewm',
    'goals_conceded_per_90_1_ewm',
    'goals_conceded_per_90_3_ewm',
    'goals_conceded_per_90_5_ewm',
    'saves_per_90_1_ewm',
    'saves_per_90_3_ewm',
    'saves_per_90_5_ewm',
    'expected_goal_involvements_per_90_1_ewm',
    'expected_goal_involvements_per_90_3_ewm',
    'expected_goal_involvements_per_90_5_ewm',
    'expected_goals_conceded_per_90_1_ewm',
    'expected_goals_conceded_per_90_3_ewm',
    'expected_goals_conceded_per_90_5_ewm',
    'interceptions_per_90_1_ewm',
    'interceptions_per_90_3_ewm',
    'interceptions_per_90_5_ewm',
    'blocks_per_90_1_ewm',
    'blocks_per_90_3_ewm',
    'blocks_per_90_5_ewm',
    'clearances_per_90_1_ewm',
    'clearances_per_90_3_ewm',
    'clearances_per_90_5_ewm',
    'tackles_per_90_1_ewm',
    'tackles_per_90_3_ewm',
    'tackles_per_90_5_ewm',
    'chances_created_per_90_1_ewm',
    'chances_created_per_90_3_ewm',
    'chances_created_per_90_5_ewm',
    'goals_prevented_per_90_1_ewm',
    'goals_prevented_per_90_3_ewm',
    'goals_prevented_per_90_5_ewm',
    'sweeper_actions_per_90_1_ewm',
    'sweeper_actions_per_90_3_ewm',
    'sweeper_actions_per_90_5_ewm',
    'tackles_won_percent_per_90_1_ewm',
    'tackles_won_percent_per_90_3_ewm',
    'tackles_won_percent_per_90_5_ewm',
],
    'def': [
    'was_home',
    'value',
    'transfers_balance',
    'selected',
    'transfers_in',
    'transfers_out',
    'difficulty',
    'opponent_difficulty',
    'win_prob',
    'percentage_net_transfers',
    'elo_diff',
    'ownership_change',
    'points_rolling_1',
    'points_rolling_3',
    'points_rolling_5',
    'xP_rolling_1',
    'xP_rolling_3',
    'xP_rolling_5',
    'minutes_rolling_1',
    'minutes_rolling_3',
    'minutes_rolling_5',
    'goals_scored_rolling_1',
    'goals_scored_rolling_3',
    'goals_scored_rolling_5',
    'assists_rolling_1',
    'assists_rolling_3',
    'assists_rolling_5',
    'clean_sheets_rolling_1',
    'clean_sheets_rolling_3',
    'clean_sheets_rolling_5',
    'goals_conceded_rolling_1',
    'goals_conceded_rolling_3',
    'goals_conceded_rolling_5',
    'own_goals_rolling_1',
    'own_goals_rolling_3',
    'own_goals_rolling_5',
    'yellow_cards_rolling_1',
    'yellow_cards_rolling_3',
    'yellow_cards_rolling_5',
    'saves_rolling_1',
    'saves_rolling_3',
    'saves_rolling_5',
    'influence_rolling_1',
    'influence_rolling_3',
    'influence_rolling_5',
    'creativity_rolling_1',
    'creativity_rolling_3',
    'creativity_rolling_5',
    'threat_rolling_1',
    'threat_rolling_3',
    'threat_rolling_5',
    'ict_index_rolling_1',
    'ict_index_rolling_3',
    'ict_index_rolling_5',
    'starts_rolling_1',
    'starts_rolling_3',
    'starts_rolling_5',
    'expected_goals_rolling_1',
    'expected_goals_rolling_3',
    'expected_goals_rolling_5',
    'expected_assists_rolling_1',
    'expected_assists_rolling_3',
    'expected_assists_rolling_5',
    'expected_goal_involvements_rolling_1',
    'expected_goal_involvements_rolling_3',
    'expected_goal_involvements_rolling_5',
    'expected_goals_conceded_rolling_1',
    'expected_goals_conceded_rolling_3',
    'expected_goals_conceded_rolling_5',
    'interceptions_rolling_1',
    'interceptions_rolling_3',
    'interceptions_rolling_5',
    'blocks_rolling_1',
    'blocks_rolling_3',
    'blocks_rolling_5',
    'clearances_rolling_1',
    'clearances_rolling_3',
    'clearances_rolling_5',
    'tackles_rolling_1',
    'tackles_rolling_3',
    'tackles_rolling_5',
    'chances_created_rolling_1',
    'chances_created_rolling_3',
    'chances_created_rolling_5',
    'goals_prevented_rolling_1',
    'goals_prevented_rolling_3',
    'goals_prevented_rolling_5',
    'sweeper_actions_rolling_1',
    'sweeper_actions_rolling_3',
    'sweeper_actions_rolling_5',
    'tackles_won_percent_rolling_1',
    'tackles_won_percent_rolling_3',
    'tackles_won_percent_rolling_5',
    'team_xG_rolling_1',
    'team_xG_rolling_3',
    'team_xG_rolling_5',
    'opponent_xG_rolling_1',
    'opponent_xG_rolling_3',
    'opponent_xG_rolling_5',
    'team_xGOT_rolling_1',
    'team_xGOT_rolling_3',
    'team_xGOT_rolling_5',
    'opponent_xGOT_rolling_1',
    'opponent_xGOT_rolling_3',
    'opponent_xGOT_rolling_5',
    'shots_faced_rolling_1',
    'shots_faced_rolling_3',
    'shots_faced_rolling_5',
    'opponent_shots_faced_rolling_1',
    'opponent_shots_faced_rolling_3',
    'opponent_shots_faced_rolling_5',
    'big_chances_faced_rolling_1',
    'big_chances_faced_rolling_3',
    'big_chances_faced_rolling_5',
    'opponent_big_chances_faced_rolling_1',
    'opponent_big_chances_faced_rolling_3',
    'opponent_big_chances_faced_rolling_5',
    'total_points_per_90_rolling_1',
    'total_points_per_90_rolling_3',
    'total_points_per_90_rolling_5',
    'goals_scored_per_90_rolling_1',
    'goals_scored_per_90_rolling_3',
    'goals_scored_per_90_rolling_5',
    'assists_per_90_rolling_1',
    'assists_per_90_rolling_3',
    'assists_per_90_rolling_5',
    'clean_sheets_per_90_rolling_1',
    'clean_sheets_per_90_rolling_3',
    'clean_sheets_per_90_rolling_5',
    'goals_conceded_per_90_rolling_1',
    'goals_conceded_per_90_rolling_3',
    'goals_conceded_per_90_rolling_5',
    'saves_per_90_rolling_1',
    'saves_per_90_rolling_3',
    'saves_per_90_rolling_5',
    'expected_goals_per_90_rolling_1',
    'expected_goals_per_90_rolling_3',
    'expected_goals_per_90_rolling_5',
    'expected_assists_per_90_rolling_1',
    'expected_assists_per_90_rolling_3',
    'expected_assists_per_90_rolling_5',
    'expected_goal_involvements_per_90_rolling_1',
    'expected_goal_involvements_per_90_rolling_3',
    'expected_goal_involvements_per_90_rolling_5',
    'expected_goals_conceded_per_90_rolling_1',
    'expected_goals_conceded_per_90_rolling_3',
    'expected_goals_conceded_per_90_rolling_5',
    'interceptions_per_90_rolling_1',
    'interceptions_per_90_rolling_3',
    'interceptions_per_90_rolling_5',
    'blocks_per_90_rolling_1',
    'blocks_per_90_rolling_3',
    'blocks_per_90_rolling_5',
    'clearances_per_90_rolling_1',
    'clearances_per_90_rolling_3',
    'clearances_per_90_rolling_5',
    'tackles_per_90_rolling_1',
    'tackles_per_90_rolling_3',
    'tackles_per_90_rolling_5',
    'chances_created_per_90_rolling_1',
    'chances_created_per_90_rolling_3',
    'chances_created_per_90_rolling_5',
    'goals_prevented_per_90_rolling_1',
    'goals_prevented_per_90_rolling_3',
    'goals_prevented_per_90_rolling_5',
    'sweeper_actions_per_90_rolling_1',
    'sweeper_actions_per_90_rolling_3',
    'sweeper_actions_per_90_rolling_5',
    'tackles_won_percent_per_90_rolling_1',
    'tackles_won_percent_per_90_rolling_3',
    'tackles_won_percent_per_90_rolling_5',
    'points_1_ewm',
    'points_3_ewm',
    'points_5_ewm',
    'xP_1_ewm',
    'xP_3_ewm',
    'xP_5_ewm',
    'minutes_1_ewm',
    'minutes_3_ewm',
    'minutes_5_ewm',
    'goals_scored_1_ewm',
    'goals_scored_3_ewm',
    'goals_scored_5_ewm',
    'assists_1_ewm',
    'assists_3_ewm',
    'assists_5_ewm',
    'clean_sheets_1_ewm',
    'clean_sheets_3_ewm',
    'clean_sheets_5_ewm',
    'goals_conceded_1_ewm',
    'goals_conceded_3_ewm',
    'goals_conceded_5_ewm',
    'own_goals_1_ewm',
    'own_goals_3_ewm',
    'own_goals_5_ewm',
    'yellow_cards_1_ewm',
    'yellow_cards_3_ewm',
    'yellow_cards_5_ewm',
    'saves_1_ewm',
    'saves_3_ewm',
    'saves_5_ewm',
    'influence_1_ewm',
    'influence_3_ewm',
    'influence_5_ewm',
    'creativity_1_ewm',
    'creativity_3_ewm',
    'creativity_5_ewm',
    'threat_1_ewm',
    'threat_3_ewm',
    'threat_5_ewm',
    'ict_index_1_ewm',
    'ict_index_3_ewm',
    'ict_index_5_ewm',
    'starts_1_ewm',
    'starts_3_ewm',
    'starts_5_ewm',
    'expected_goals_1_ewm',
    'expected_goals_3_ewm',
    'expected_goals_5_ewm',
    'expected_assists_1_ewm',
    'expected_assists_3_ewm',
    'expected_assists_5_ewm',
    'expected_goal_involvements_1_ewm',
    'expected_goal_involvements_3_ewm',
    'expected_goal_involvements_5_ewm',
    'expected_goals_conceded_1_ewm',
    'expected_goals_conceded_3_ewm',
    'expected_goals_conceded_5_ewm',
    'interceptions_1_ewm',
    'interceptions_3_ewm',
    'interceptions_5_ewm',
    'blocks_1_ewm',
    'blocks_3_ewm',
    'blocks_5_ewm',
    'clearances_1_ewm',
    'clearances_3_ewm',
    'clearances_5_ewm',
    'tackles_1_ewm',
    'tackles_3_ewm',
    'tackles_5_ewm',
    'chances_created_1_ewm',
    'chances_created_3_ewm',
    'chances_created_5_ewm',
    'goals_prevented_1_ewm',
    'goals_prevented_3_ewm',
    'goals_prevented_5_ewm',
    'sweeper_actions_1_ewm',
    'sweeper_actions_3_ewm',
    'sweeper_actions_5_ewm',
    'tackles_won_percent_1_ewm',
    'tackles_won_percent_3_ewm',
    'tackles_won_percent_5_ewm',
    'team_xG_1_ewm',
    'team_xG_3_ewm',
    'team_xG_5_ewm',
    'opponent_xG_1_ewm',
    'opponent_xG_3_ewm',
    'opponent_xG_5_ewm',
    'team_xGOT_1_ewm',
    'team_xGOT_3_ewm',
    'team_xGOT_5_ewm',
    'opponent_xGOT_1_ewm',
    'opponent_xGOT_3_ewm',
    'opponent_xGOT_5_ewm',
    'shots_faced_1_ewm',
    'shots_faced_3_ewm',
    'shots_faced_5_ewm',
    'opponent_shots_faced_1_ewm',
    'opponent_shots_faced_3_ewm',
    'opponent_shots_faced_5_ewm',
    'big_chances_faced_1_ewm',
    'big_chances_faced_3_ewm',
    'big_chances_faced_5_ewm',
    'opponent_big_chances_faced_1_ewm',
    'opponent_big_chances_faced_3_ewm',
    'opponent_big_chances_faced_5_ewm',
    'total_points_per_90_1_ewm',
    'total_points_per_90_3_ewm',
    'total_points_per_90_5_ewm',
    'goals_scored_per_90_1_ewm',
    'goals_scored_per_90_3_ewm',
    'goals_scored_per_90_5_ewm',
    'assists_per_90_1_ewm',
    'assists_per_90_3_ewm',
    'assists_per_90_5_ewm',
    'clean_sheets_per_90_1_ewm',
    'clean_sheets_per_90_3_ewm',
    'clean_sheets_per_90_5_ewm',
    'goals_conceded_per_90_1_ewm',
    'goals_conceded_per_90_3_ewm',
    'goals_conceded_per_90_5_ewm',
    'saves_per_90_1_ewm',
    'saves_per_90_3_ewm',
    'saves_per_90_5_ewm',
    'expected_goals_per_90_1_ewm',
    'expected_goals_per_90_3_ewm',
    'expected_goals_per_90_5_ewm',
    'expected_assists_per_90_1_ewm',
    'expected_assists_per_90_3_ewm',
    'expected_assists_per_90_5_ewm',
    'expected_goal_involvements_per_90_1_ewm',
    'expected_goal_involvements_per_90_3_ewm',
    'expected_goal_involvements_per_90_5_ewm',
    'expected_goals_conceded_per_90_1_ewm',
    'expected_goals_conceded_per_90_3_ewm',
    'expected_goals_conceded_per_90_5_ewm',
    'interceptions_per_90_1_ewm',
    'interceptions_per_90_3_ewm',
    'interceptions_per_90_5_ewm',
    'blocks_per_90_1_ewm',
    'blocks_per_90_3_ewm',
    'blocks_per_90_5_ewm',
    'clearances_per_90_1_ewm',
    'clearances_per_90_3_ewm',
    'clearances_per_90_5_ewm',
    'tackles_per_90_1_ewm',
    'tackles_per_90_3_ewm',
    'tackles_per_90_5_ewm',
    'chances_created_per_90_1_ewm',
    'chances_created_per_90_3_ewm',
    'chances_created_per_90_5_ewm',
    'goals_prevented_per_90_1_ewm',
    'goals_prevented_per_90_3_ewm',
    'goals_prevented_per_90_5_ewm',
    'sweeper_actions_per_90_1_ewm',
    'sweeper_actions_per_90_3_ewm',
    'sweeper_actions_per_90_5_ewm',
    'tackles_won_percent_per_90_1_ewm',
    'tackles_won_percent_per_90_3_ewm',
    'tackles_won_percent_per_90_5_ewm',
],
    'mid': [
    'was_home',
    'value',
    'transfers_balance',
    'selected',
    'transfers_in',
    'transfers_out',
    'difficulty',
    'opponent_difficulty',
    'win_prob',
    'percentage_net_transfers',
    'elo_diff',
    'ownership_change',
    'points_rolling_1',
    'points_rolling_3',
    'points_rolling_5',
    'xP_rolling_1',
    'xP_rolling_3',
    'xP_rolling_5',
    'minutes_rolling_1',
    'minutes_rolling_3',
    'minutes_rolling_5',
    'goals_scored_rolling_1',
    'goals_scored_rolling_3',
    'goals_scored_rolling_5',
    'assists_rolling_1',
    'assists_rolling_3',
    'assists_rolling_5',
    'clean_sheets_rolling_1',
    'clean_sheets_rolling_3',
    'clean_sheets_rolling_5',
    'goals_conceded_rolling_1',
    'goals_conceded_rolling_3',
    'goals_conceded_rolling_5',
    'own_goals_rolling_1',
    'own_goals_rolling_3',
    'own_goals_rolling_5',
    'yellow_cards_rolling_1',
    'yellow_cards_rolling_3',
    'yellow_cards_rolling_5',
    'saves_rolling_1',
    'saves_rolling_3',
    'saves_rolling_5',
    'influence_rolling_1',
    'influence_rolling_3',
    'influence_rolling_5',
    'creativity_rolling_1',
    'creativity_rolling_3',
    'creativity_rolling_5',
    'threat_rolling_1',
    'threat_rolling_3',
    'threat_rolling_5',
    'ict_index_rolling_1',
    'ict_index_rolling_3',
    'ict_index_rolling_5',
    'starts_rolling_1',
    'starts_rolling_3',
    'starts_rolling_5',
    'expected_goals_rolling_1',
    'expected_goals_rolling_3',
    'expected_goals_rolling_5',
    'expected_assists_rolling_1',
    'expected_assists_rolling_3',
    'expected_assists_rolling_5',
    'expected_goal_involvements_rolling_1',
    'expected_goal_involvements_rolling_3',
    'expected_goal_involvements_rolling_5',
    'expected_goals_conceded_rolling_1',
    'expected_goals_conceded_rolling_3',
    'expected_goals_conceded_rolling_5',
    'interceptions_rolling_1',
    'interceptions_rolling_3',
    'interceptions_rolling_5',
    'blocks_rolling_1',
    'blocks_rolling_3',
    'blocks_rolling_5',
    'clearances_rolling_1',
    'clearances_rolling_3',
    'clearances_rolling_5',
    'tackles_rolling_1',
    'tackles_rolling_3',
    'tackles_rolling_5',
    'chances_created_rolling_1',
    'chances_created_rolling_3',
    'chances_created_rolling_5',
    'goals_prevented_rolling_1',
    'goals_prevented_rolling_3',
    'goals_prevented_rolling_5',
    'sweeper_actions_rolling_1',
    'sweeper_actions_rolling_3',
    'sweeper_actions_rolling_5',
    'tackles_won_percent_rolling_1',
    'tackles_won_percent_rolling_3',
    'tackles_won_percent_rolling_5',
    'team_xG_rolling_1',
    'team_xG_rolling_3',
    'team_xG_rolling_5',
    'opponent_xG_rolling_1',
    'opponent_xG_rolling_3',
    'opponent_xG_rolling_5',
    'team_xGOT_rolling_1',
    'team_xGOT_rolling_3',
    'team_xGOT_rolling_5',
    'opponent_xGOT_rolling_1',
    'opponent_xGOT_rolling_3',
    'opponent_xGOT_rolling_5',
    'shots_faced_rolling_1',
    'shots_faced_rolling_3',
    'shots_faced_rolling_5',
    'opponent_shots_faced_rolling_1',
    'opponent_shots_faced_rolling_3',
    'opponent_shots_faced_rolling_5',
    'big_chances_faced_rolling_1',
    'big_chances_faced_rolling_3',
    'big_chances_faced_rolling_5',
    'opponent_big_chances_faced_rolling_1',
    'opponent_big_chances_faced_rolling_3',
    'opponent_big_chances_faced_rolling_5',
    'total_points_per_90_rolling_1',
    'total_points_per_90_rolling_3',
    'total_points_per_90_rolling_5',
    'goals_scored_per_90_rolling_1',
    'goals_scored_per_90_rolling_3',
    'goals_scored_per_90_rolling_5',
    'assists_per_90_rolling_1',
    'assists_per_90_rolling_3',
    'assists_per_90_rolling_5',
    'clean_sheets_per_90_rolling_1',
    'clean_sheets_per_90_rolling_3',
    'clean_sheets_per_90_rolling_5',
    'goals_conceded_per_90_rolling_1',
    'goals_conceded_per_90_rolling_3',
    'goals_conceded_per_90_rolling_5',
    'saves_per_90_rolling_1',
    'saves_per_90_rolling_3',
    'saves_per_90_rolling_5',
    'expected_goals_per_90_rolling_1',
    'expected_goals_per_90_rolling_3',
    'expected_goals_per_90_rolling_5',
    'expected_assists_per_90_rolling_1',
    'expected_assists_per_90_rolling_3',
    'expected_assists_per_90_rolling_5',
    'expected_goal_involvements_per_90_rolling_1',
    'expected_goal_involvements_per_90_rolling_3',
    'expected_goal_involvements_per_90_rolling_5',
    'expected_goals_conceded_per_90_rolling_1',
    'expected_goals_conceded_per_90_rolling_3',
    'expected_goals_conceded_per_90_rolling_5',
    'interceptions_per_90_rolling_1',
    'interceptions_per_90_rolling_3',
    'interceptions_per_90_rolling_5',
    'blocks_per_90_rolling_1',
    'blocks_per_90_rolling_3',
    'blocks_per_90_rolling_5',
    'clearances_per_90_rolling_1',
    'clearances_per_90_rolling_3',
    'clearances_per_90_rolling_5',
    'tackles_per_90_rolling_1',
    'tackles_per_90_rolling_3',
    'tackles_per_90_rolling_5',
    'chances_created_per_90_rolling_1',
    'chances_created_per_90_rolling_3',
    'chances_created_per_90_rolling_5',
    'goals_prevented_per_90_rolling_1',
    'goals_prevented_per_90_rolling_3',
    'goals_prevented_per_90_rolling_5',
    'sweeper_actions_per_90_rolling_1',
    'sweeper_actions_per_90_rolling_3',
    'sweeper_actions_per_90_rolling_5',
    'tackles_won_percent_per_90_rolling_1',
    'tackles_won_percent_per_90_rolling_3',
    'tackles_won_percent_per_90_rolling_5',
    'points_1_ewm',
    'points_3_ewm',
    'points_5_ewm',
    'xP_1_ewm',
    'xP_3_ewm',
    'xP_5_ewm',
    'minutes_1_ewm',
    'minutes_3_ewm',
    'minutes_5_ewm',
    'goals_scored_1_ewm',
    'goals_scored_3_ewm',
    'goals_scored_5_ewm',
    'assists_1_ewm',
    'assists_3_ewm',
    'assists_5_ewm',
    'clean_sheets_1_ewm',
    'clean_sheets_3_ewm',
    'clean_sheets_5_ewm',
    'goals_conceded_1_ewm',
    'goals_conceded_3_ewm',
    'goals_conceded_5_ewm',
    'own_goals_1_ewm',
    'own_goals_3_ewm',
    'own_goals_5_ewm',
    'yellow_cards_1_ewm',
    'yellow_cards_3_ewm',
    'yellow_cards_5_ewm',
    'saves_1_ewm',
    'saves_3_ewm',
    'saves_5_ewm',
    'influence_1_ewm',
    'influence_3_ewm',
    'influence_5_ewm',
    'creativity_1_ewm',
    'creativity_3_ewm',
    'creativity_5_ewm',
    'threat_1_ewm',
    'threat_3_ewm',
    'threat_5_ewm',
    'ict_index_1_ewm',
    'ict_index_3_ewm',
    'ict_index_5_ewm',
    'starts_1_ewm',
    'starts_3_ewm',
    'starts_5_ewm',
    'expected_goals_1_ewm',
    'expected_goals_3_ewm',
    'expected_goals_5_ewm',
    'expected_assists_1_ewm',
    'expected_assists_3_ewm',
    'expected_assists_5_ewm',
    'expected_goal_involvements_1_ewm',
    'expected_goal_involvements_3_ewm',
    'expected_goal_involvements_5_ewm',
    'expected_goals_conceded_1_ewm',
    'expected_goals_conceded_3_ewm',
    'expected_goals_conceded_5_ewm',
    'interceptions_1_ewm',
    'interceptions_3_ewm',
    'interceptions_5_ewm',
    'blocks_1_ewm',
    'blocks_3_ewm',
    'blocks_5_ewm',
    'clearances_1_ewm',
    'clearances_3_ewm',
    'clearances_5_ewm',
    'tackles_1_ewm',
    'tackles_3_ewm',
    'tackles_5_ewm',
    'chances_created_1_ewm',
    'chances_created_3_ewm',
    'chances_created_5_ewm',
    'goals_prevented_1_ewm',
    'goals_prevented_3_ewm',
    'goals_prevented_5_ewm',
    'sweeper_actions_1_ewm',
    'sweeper_actions_3_ewm',
    'sweeper_actions_5_ewm',
    'tackles_won_percent_1_ewm',
    'tackles_won_percent_3_ewm',
    'tackles_won_percent_5_ewm',
    'team_xG_1_ewm',
    'team_xG_3_ewm',
    'team_xG_5_ewm',
    'opponent_xG_1_ewm',
    'opponent_xG_3_ewm',
    'opponent_xG_5_ewm',
    'team_xGOT_1_ewm',
    'team_xGOT_3_ewm',
    'team_xGOT_5_ewm',
    'opponent_xGOT_1_ewm',
    'opponent_xGOT_3_ewm',
    'opponent_xGOT_5_ewm',
    'shots_faced_1_ewm',
    'shots_faced_3_ewm',
    'shots_faced_5_ewm',
    'opponent_shots_faced_1_ewm',
    'opponent_shots_faced_3_ewm',
    'opponent_shots_faced_5_ewm',
    'big_chances_faced_1_ewm',
    'big_chances_faced_3_ewm',
    'big_chances_faced_5_ewm',
    'opponent_big_chances_faced_1_ewm',
    'opponent_big_chances_faced_3_ewm',
    'opponent_big_chances_faced_5_ewm',
    'total_points_per_90_1_ewm',
    'total_points_per_90_3_ewm',
    'total_points_per_90_5_ewm',
    'goals_scored_per_90_1_ewm',
    'goals_scored_per_90_3_ewm',
    'goals_scored_per_90_5_ewm',
    'assists_per_90_1_ewm',
    'assists_per_90_3_ewm',
    'assists_per_90_5_ewm',
    'clean_sheets_per_90_1_ewm',
    'clean_sheets_per_90_3_ewm',
    'clean_sheets_per_90_5_ewm',
    'goals_conceded_per_90_1_ewm',
    'goals_conceded_per_90_3_ewm',
    'goals_conceded_per_90_5_ewm',
    'saves_per_90_1_ewm',
    'saves_per_90_3_ewm',
    'saves_per_90_5_ewm',
    'expected_goals_per_90_1_ewm',
    'expected_goals_per_90_3_ewm',
    'expected_goals_per_90_5_ewm',
    'expected_assists_per_90_1_ewm',
    'expected_assists_per_90_3_ewm',
    'expected_assists_per_90_5_ewm',
    'expected_goal_involvements_per_90_1_ewm',
    'expected_goal_involvements_per_90_3_ewm',
    'expected_goal_involvements_per_90_5_ewm',
    'expected_goals_conceded_per_90_1_ewm',
    'expected_goals_conceded_per_90_3_ewm',
    'expected_goals_conceded_per_90_5_ewm',
    'interceptions_per_90_1_ewm',
    'interceptions_per_90_3_ewm',
    'interceptions_per_90_5_ewm',
    'blocks_per_90_1_ewm',
    'blocks_per_90_3_ewm',
    'blocks_per_90_5_ewm',
    'clearances_per_90_1_ewm',
    'clearances_per_90_3_ewm',
    'clearances_per_90_5_ewm',
    'tackles_per_90_1_ewm',
    'tackles_per_90_3_ewm',
    'tackles_per_90_5_ewm',
    'chances_created_per_90_1_ewm',
    'chances_created_per_90_3_ewm',
    'chances_created_per_90_5_ewm',
    'goals_prevented_per_90_1_ewm',
    'goals_prevented_per_90_3_ewm',
    'goals_prevented_per_90_5_ewm',
    'sweeper_actions_per_90_1_ewm',
    'sweeper_actions_per_90_3_ewm',
    'sweeper_actions_per_90_5_ewm',
    'tackles_won_percent_per_90_1_ewm',
    'tackles_won_percent_per_90_3_ewm',
    'tackles_won_percent_per_90_5_ewm',
    'recoveries_rolling_1',
    'recoveries_rolling_3',
    'recoveries_rolling_5',
    'recoveries_per_90_rolling_1',
    'recoveries_per_90_rolling_3',
    'recoveries_per_90_rolling_5',
    'recoveries_1_ewm',
    'recoveries_3_ewm',
    'recoveries_5_ewm',
    'recoveries_per_90_1_ewm',
    'recoveries_per_90_3_ewm',
    'recoveries_per_90_5_ewm',
],
    'fwd': [
    'was_home',
    'value',
    'transfers_balance',
    'selected',
    'transfers_in',
    'transfers_out',
    'difficulty',
    'opponent_difficulty',
    'win_prob',
    'percentage_net_transfers',
    'elo_diff',
    'ownership_change',
    'points_rolling_1',
    'points_rolling_3',
    'points_rolling_5',
    'xP_rolling_1',
    'xP_rolling_3',
    'xP_rolling_5',
    'minutes_rolling_1',
    'minutes_rolling_3',
    'minutes_rolling_5',
    'goals_scored_rolling_1',
    'goals_scored_rolling_3',
    'goals_scored_rolling_5',
    'assists_rolling_1',
    'assists_rolling_3',
    'assists_rolling_5',
    'yellow_cards_rolling_1',
    'yellow_cards_rolling_3',
    'yellow_cards_rolling_5',
    'influence_rolling_1',
    'influence_rolling_3',
    'influence_rolling_5',
    'creativity_rolling_1',
    'creativity_rolling_3',
    'creativity_rolling_5',
    'threat_rolling_1',
    'threat_rolling_3',
    'threat_rolling_5',
    'ict_index_rolling_1',
    'ict_index_rolling_3',
    'ict_index_rolling_5',
    'starts_rolling_1',
    'starts_rolling_3',
    'starts_rolling_5',
    'expected_goals_rolling_1',
    'expected_goals_rolling_3',
    'expected_goals_rolling_5',
    'expected_assists_rolling_1',
    'expected_assists_rolling_3',
    'expected_assists_rolling_5',
    'expected_goal_involvements_rolling_1',
    'expected_goal_involvements_rolling_3',
    'expected_goal_involvements_rolling_5',
    'interceptions_rolling_1',
    'interceptions_rolling_3',
    'interceptions_rolling_5',
    'blocks_rolling_1',
    'blocks_rolling_3',
    'blocks_rolling_5',
    'clearances_rolling_1',
    'clearances_rolling_3',
    'clearances_rolling_5',
    'tackles_rolling_1',
    'tackles_rolling_3',
    'tackles_rolling_5',
    'chances_created_rolling_1',
    'chances_created_rolling_3',
    'chances_created_rolling_5',
    'goals_prevented_rolling_1',
    'goals_prevented_rolling_3',
    'goals_prevented_rolling_5',
    'sweeper_actions_rolling_1',
    'sweeper_actions_rolling_3',
    'sweeper_actions_rolling_5',
    'tackles_won_percent_rolling_1',
    'tackles_won_percent_rolling_3',
    'tackles_won_percent_rolling_5',
    'team_xG_rolling_1',
    'team_xG_rolling_3',
    'team_xG_rolling_5',
    'team_xGOT_rolling_1',
    'team_xGOT_rolling_3',
    'team_xGOT_rolling_5',
    'opponent_shots_faced_rolling_1',
    'opponent_shots_faced_rolling_3',
    'opponent_shots_faced_rolling_5',
    'big_chances_faced_rolling_1',
    'big_chances_faced_rolling_3',
    'big_chances_faced_rolling_5',
    'total_points_per_90_rolling_1',
    'total_points_per_90_rolling_3',
    'total_points_per_90_rolling_5',
    'goals_scored_per_90_rolling_1',
    'goals_scored_per_90_rolling_3',
    'goals_scored_per_90_rolling_5',
    'assists_per_90_rolling_1',
    'assists_per_90_rolling_3',
    'assists_per_90_rolling_5',
    'expected_goals_per_90_rolling_1',
    'expected_goals_per_90_rolling_3',
    'expected_goals_per_90_rolling_5',
    'expected_assists_per_90_rolling_1',
    'expected_assists_per_90_rolling_3',
    'expected_assists_per_90_rolling_5',
    'expected_goal_involvements_per_90_rolling_1',
    'expected_goal_involvements_per_90_rolling_3',
    'expected_goal_involvements_per_90_rolling_5',
    'interceptions_per_90_rolling_1',
    'interceptions_per_90_rolling_3',
    'interceptions_per_90_rolling_5',
    'blocks_per_90_rolling_1',
    'blocks_per_90_rolling_3',
    'blocks_per_90_rolling_5',
    'clearances_per_90_rolling_1',
    'clearances_per_90_rolling_3',
    'clearances_per_90_rolling_5',
    'tackles_per_90_rolling_1',
    'tackles_per_90_rolling_3',
    'tackles_per_90_rolling_5',
    'chances_created_per_90_rolling_1',
    'chances_created_per_90_rolling_3',
    'chances_created_per_90_rolling_5',
    'goals_prevented_per_90_rolling_1',
    'goals_prevented_per_90_rolling_3',
    'goals_prevented_per_90_rolling_5',
    'sweeper_actions_per_90_rolling_1',
    'sweeper_actions_per_90_rolling_3',
    'sweeper_actions_per_90_rolling_5',
    'tackles_won_percent_per_90_rolling_1',
    'tackles_won_percent_per_90_rolling_3',
    'tackles_won_percent_per_90_rolling_5',
    'points_1_ewm',
    'points_3_ewm',
    'points_5_ewm',
    'xP_1_ewm',
    'xP_3_ewm',
    'xP_5_ewm',
    'minutes_1_ewm',
    'minutes_3_ewm',
    'minutes_5_ewm',
    'goals_scored_1_ewm',
    'goals_scored_3_ewm',
    'goals_scored_5_ewm',
    'assists_1_ewm',
    'assists_3_ewm',
    'assists_5_ewm',
    'yellow_cards_1_ewm',
    'yellow_cards_3_ewm',
    'yellow_cards_5_ewm',
    'influence_1_ewm',
    'influence_3_ewm',
    'influence_5_ewm',
    'creativity_1_ewm',
    'creativity_3_ewm',
    'creativity_5_ewm',
    'threat_1_ewm',
    'threat_3_ewm',
    'threat_5_ewm',
    'ict_index_1_ewm',
    'ict_index_3_ewm',
    'ict_index_5_ewm',
    'starts_1_ewm',
    'starts_3_ewm',
    'starts_5_ewm',
    'expected_goals_1_ewm',
    'expected_goals_3_ewm',
    'expected_goals_5_ewm',
    'expected_assists_1_ewm',
    'expected_assists_3_ewm',
    'expected_assists_5_ewm',
    'expected_goal_involvements_1_ewm',
    'expected_goal_involvements_3_ewm',
    'expected_goal_involvements_5_ewm',
    'interceptions_1_ewm',
    'interceptions_3_ewm',
    'interceptions_5_ewm',
    'blocks_1_ewm',
    'blocks_3_ewm',
    'blocks_5_ewm',
    'clearances_1_ewm',
    'clearances_3_ewm',
    'clearances_5_ewm',
    'tackles_1_ewm',
    'tackles_3_ewm',
    'tackles_5_ewm',
    'chances_created_1_ewm',
    'chances_created_3_ewm',
    'chances_created_5_ewm',
    'goals_prevented_1_ewm',
    'goals_prevented_3_ewm',
    'goals_prevented_5_ewm',
    'sweeper_actions_1_ewm',
    'sweeper_actions_3_ewm',
    'sweeper_actions_5_ewm',
    'tackles_won_percent_1_ewm',
    'tackles_won_percent_3_ewm',
    'tackles_won_percent_5_ewm',
    'team_xG_1_ewm',
    'team_xG_3_ewm',
    'team_xG_5_ewm',
    'opponent_xG_1_ewm',
    'opponent_xG_3_ewm',
    'opponent_xG_5_ewm',
    'team_xGOT_1_ewm',
    'team_xGOT_3_ewm',
    'team_xGOT_5_ewm',
    'opponent_xGOT_1_ewm',
    'opponent_xGOT_3_ewm',
    'opponent_xGOT_5_ewm',
    'opponent_shots_faced_1_ewm',
    'opponent_shots_faced_3_ewm',
    'opponent_shots_faced_5_ewm',
    'big_chances_faced_1_ewm',
    'big_chances_faced_3_ewm',
    'big_chances_faced_5_ewm',
    'opponent_big_chances_faced_1_ewm',
    'opponent_big_chances_faced_3_ewm',
    'opponent_big_chances_faced_5_ewm',
    'total_points_per_90_1_ewm',
    'total_points_per_90_3_ewm',
    'total_points_per_90_5_ewm',
    'goals_scored_per_90_1_ewm',
    'goals_scored_per_90_3_ewm',
    'goals_scored_per_90_5_ewm',
    'assists_per_90_1_ewm',
    'assists_per_90_3_ewm',
    'assists_per_90_5_ewm',
    'expected_goals_per_90_1_ewm',
    'expected_goals_per_90_3_ewm',
    'expected_goals_per_90_5_ewm',
    'expected_assists_per_90_1_ewm',
    'expected_assists_per_90_3_ewm',
    'expected_assists_per_90_5_ewm',
    'expected_goal_involvements_per_90_1_ewm',
    'expected_goal_involvements_per_90_3_ewm',
    'expected_goal_involvements_per_90_5_ewm',
    'interceptions_per_90_1_ewm',
    'interceptions_per_90_3_ewm',
    'interceptions_per_90_5_ewm',
    'blocks_per_90_1_ewm',
    'blocks_per_90_3_ewm',
    'blocks_per_90_5_ewm',
    'clearances_per_90_1_ewm',
    'clearances_per_90_3_ewm',
    'clearances_per_90_5_ewm',
    'tackles_per_90_1_ewm',
    'tackles_per_90_3_ewm',
    'tackles_per_90_5_ewm',
    'chances_created_per_90_1_ewm',
    'chances_created_per_90_3_ewm',
    'chances_created_per_90_5_ewm',
    'goals_prevented_per_90_1_ewm',
    'goals_prevented_per_90_3_ewm',
    'goals_prevented_per_90_5_ewm',
    'sweeper_actions_per_90_1_ewm',
    'sweeper_actions_per_90_3_ewm',
    'sweeper_actions_per_90_5_ewm',
    'tackles_won_percent_per_90_1_ewm',
    'tackles_won_percent_per_90_3_ewm',
    'tackles_won_percent_per_90_5_ewm',
],
}

## Train + predict, per position


In [ ]:
def train_position(suffix, position, blank_gw_rounds=(1,2), n_iter=20,
                    cv_window=10, retune_every=5, verbose=True):
    """Feature selection -> model comparison/tuning -> walk-forward predictions for one
    position, mirroring `xP_predictor - {pos}.ipynb` + `get predictions.ipynb` combined, but
    generalized to whichever `TARGET_COL` is configured above (originally written for a
    `total_points`-targeted run, reused unchanged for the `xP`-targeted run this notebook
    currently performs). Saves `hist/feats_{suffix}_{OUTPUT_SUFFIX}`,
    `models_by_round_{suffix}_{OUTPUT_SUFFIX}`, and `hist/{suffix}_preds_{OUTPUT_SUFFIX}.csv`;
    returns (preds_df, mae_by_round, r2_by_round) for a quick sanity check.

    `cv_window`/`retune_every` exist because the original (uncapped) design blew up: by
    round 28 alone, `tune_best_model`'s CV folds (`val_gws=range(1, round)`) had grown to
    26, giving 26 folds x 40 candidates = 1040 fits for that ONE round, and it only gets
    worse for every round after -- observed directly running this. `cv_window` caps both
    the model-comparison averaging window and the CV fold count to the most recent
    `cv_window` gameweeks instead of all of history (a player's current form is what
    matters for picking a model anyway, not gameweek-1 data 30+ rounds back).
    `retune_every` skips the full comparison+search on most rounds, reusing the last
    selected model/hyperparameters and just refitting fresh on the growing training
    window (which the walk-forward prediction loop below does regardless) -- real
    walk-forward systems don't usually re-derive "best model family + hyperparameters"
    every single week either, since that's fairly stable week-to-week."""
    data = player_data[player_data['position'] == position]
    avail_feats = AVAIL_FEATS_BY_POS[suffix]

    # Only TRAIN (feature selection, model comparison/tuning, and the walk-forward
    # refits below) on rows where the player has actually been playing recently
    # (minutes_rolling_3 > 0, i.e. some minutes in at least one of their last 3
    # gameweeks) -- a player out of the squad for a long stretch (long-term injury,
    # out of favor, permanent backup) contributes mostly-zero rows that dilute the
    # correlation/model-fitting signal without saying anything about how an ACTIVE
    # player scores. Checked directly on the regenerated data (rolling averages now
    # use min_periods=1, see data.ipynb): this drops to 33.1% (GK) / 53.2% (DEF) /
    # 57.8% (MID) / 51.6% (FWD) of rows. Test/prediction rows are deliberately NOT
    # filtered (see the walk-forward loop below), so `hist/{suffix}_preds_{OUTPUT_SUFFIX}.csv`
    # still covers every player in the universe regardless of recent minutes -- only
    # what the model learns FROM changes.
    #
    # NaN minutes_rolling_3 (no history at all yet -- now only true at round 1, since
    # the rolling averages were regenerated to use whatever's available instead of
    # requiring a full window) is treated as "unknown, give the benefit of the doubt"
    # rather than "inactive". Round-1 rows that pass this still have NaN in every OTHER
    # rolling/ewm feature (nobody has any history at round 1), which sklearn can't fit
    # on -- fillna(0) below covers that; it's only reached for round 1 now, not the
    # multi-round wipeout the pre-regeneration data would have caused.
    active_mask = data['minutes_rolling_3'].notna() | (data['minutes_rolling_5'] > 0)
    data_train_pool = data[active_mask].fillna(0)

    # --- feature selection per round ---
    feats = {}
    for round_ in range(6, 39):
        curr = data_train_pool[(data_train_pool['round'] > 3) & (data_train_pool['round'] < round_)].fillna(0)
        feats[round_] = select_features_by_correlation(
            curr, avail_feats, target_col=TARGET_COL, min_corr=0.1,
            redundancy_threshold=0.5, verbose=False,
        )
    joblib.dump(feats, f'./hist/feats_{suffix}_{OUTPUT_SUFFIX}')
    if verbose:
        print(f"[{suffix}] feature selection done for {len(feats)} rounds")

    # --- model comparison + tuning per round (capped window, periodic retuning) ---
    models_by_round = {}
    last_selection = None
    for round_ in range(5, 38):
        feats_round = feats[round_ + 1] + ['round', 'element', 'was_home', TARGET_COL]

        if last_selection is None or (round_ - 5) % retune_every == 0:
            gw_lo = max(3, round_ - cv_window + 1)
            combined, summary = evaluate_across_gws(
                data_train_pool[feats_round], gw_list=range(gw_lo, round_ + 1),
                target_col=TARGET_COL, drop_cols=DROP_COLS,
            )
            base_name = summary.sort_values("MAE").iloc[0]["model"]
            base_mae = summary.sort_values("MAE").iloc[0]["MAE"]
            base_r2 = summary.sort_values("MAE").iloc[0]["R2"]

            val_lo = max(1, round_ - cv_window)
            best_model, best_estimator, search, c_final, mae, rmse, r2 = tune_best_model(
                data_train_pool[feats_round], summary, val_gws=range(val_lo, round_), final_test_gw=round_ + 1,
                target_col=TARGET_COL, drop_cols=DROP_COLS, n_iter=n_iter,
            )

            if mae > base_mae:
                # `search` (the full RandomizedSearchCV object, with per-fold CV results)
                # is deliberately NOT kept here -- nothing downstream reads it, and
                # keeping it around bloated a single position's joblib dump past 100MB
                # for no benefit.
                last_selection = {
                    'mae': mae, 'r2': r2, 'best_model': best_model, 'c_final': c_final,
                    'best_estimator': best_estimator,
                }
            else:
                last_selection = {
                    'mae': base_mae, 'r2': base_r2, 'best_model': base_name, 'c_final': c_final,
                }
            if verbose:
                print(f"[{suffix}] round {round_ + 1}: RETUNED, best={last_selection['best_model']} "
                      f"MAE={last_selection['mae']:.3f}")
        elif verbose:
            print(f"[{suffix}] round {round_ + 1}: reusing last tuned selection "
                  f"({last_selection['best_model']})")

        models_by_round[round_ + 1] = last_selection
    joblib.dump(models_by_round, f'./models_by_round_{suffix}_{OUTPUT_SUFFIX}')

    # --- walk-forward predictions, same structure as get predictions.ipynb ---
    preds_df = pd.DataFrame()
    mae_by_round, r2_by_round = {}, {}
    cached_model = None

    for ro in range(6, 39):
        tar_feats = feats[ro].copy()

        # Train on active players only; test/predict on the FULL population for this
        # round (everyone needs a prediction downstream, whether or not they've
        # recently played).
        X_train = data_train_pool[data_train_pool['round'] < ro][tar_feats]
        y_train = data_train_pool[data_train_pool['round'] < ro][TARGET_COL]
        c = abs(y_train.min()) + 1.0
        y_train_shifted = y_train + c

        X_test = data[data['round'] == ro][tar_feats]
        X_test_index = data[data['round'] == ro]['element']
        X_test_names = data[data['round'] == ro]['name']
        y_test = data[data['round'] == ro][TARGET_COL]

        if ro in blank_gw_rounds:
            # blank-gameweek rounds: not enough fixtures that week to retrain sensibly,
            # so reuse last round's fitted model on this round's (smaller) feature set,
            # same special-casing as the xP predictors.
            model = cached_model
            tar_feats = feats[ro - 1].copy()
            X_test = data[data['round'] == ro][tar_feats]
            X_test_index = data[data['round'] == ro]['element']
            X_test_names = data[data['round'] == ro]['name']
            y_test = data[data['round'] == ro][TARGET_COL]
        else:
            model_info = models_by_round[ro]
            if 'best_estimator' in model_info:
                model = model_info['best_estimator'].regressor  # not fitted yet
            else:
                model = RandomForestRegressor()
            model.fit(X_train, y_train_shifted)

        target_preds = model.predict(X_test) - c

        preds = {
            'element': X_test_index.values, 'name': X_test_names.values,
            f'{TARGET_COL}_pred': target_preds, f'{TARGET_COL}_actual': y_test.values, 'round': ro,
        }
        mae = mean_absolute_error(y_test, target_preds)
        rmse = np.sqrt(mean_squared_error(y_test, target_preds))
        r2 = r2_score(y_test, target_preds)
        mae_by_round[ro], r2_by_round[ro] = mae, r2
        if verbose:
            print(f"[{suffix}] round {ro} - MAE: {mae:.4f}, RMSE: {rmse:.4f}, R2: {r2:.4f}")

        preds_df = pd.concat([preds_df, pd.DataFrame(preds)], ignore_index=True)
        cached_model = model

    preds_df.to_csv(f'./hist/{suffix}_preds_{OUTPUT_SUFFIX}.csv', index=False)
    return preds_df, mae_by_round, r2_by_round


In [ ]:
results = {}
for suffix, position in POSITIONS.items():
    print(f"\n=== {position} ===")
    results[suffix] = train_position(suffix, position)


### Sanity check: active-player-filtered R2 vs the original (unfiltered) xP model


In [ ]:
summary_rows = []
for suffix, position in POSITIONS.items():
    preds_df, mae_by_round, r2_by_round = results[suffix]
    orig_preds = pd.read_csv(f'./hist/{suffix}_preds.csv')  # original xP-targeted predictions, no active-player filter

    new_r2 = r2_score(preds_df[f'{TARGET_COL}_actual'], preds_df[f'{TARGET_COL}_pred'])
    orig_r2 = r2_score(orig_preds['xP_actual'], orig_preds['xP_pred'])

    summary_rows.append({
        'position': position,
        f'R2 (active-filtered {TARGET_COL} model)': new_r2,
        'R2 (original xP model, unfiltered)': orig_r2,
    })

pd.DataFrame(summary_rows)